# 03 – Dashboard gerencial interactivo (Python + Plotly)

Dashboard de la bodega de datos NYC Taxi 2018. Lee los cuatro **cubos OLAP** de Google BigQuery (`cubo_demanda`, `cubo_ingresos`, `cubo_operacion`, `cubo_tiempo`) y genera un tablero interactivo con una sección por cada pregunta de inteligencia de negocios.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todas*. Al final se descarga el archivo `dashboard_nyc_taxi_2018.html`, que se abre con doble clic en cualquier navegador.

**Interactividad:** menús desplegables para filtrar por distrito, información al pasar el mouse sobre cada barra o celda, zoom y barra de rango en la serie diaria.

## 1. Conexión a BigQuery y lectura de los cubos
Se autentica la cuenta de Google (con acceso al proyecto `dw-taxi-nyc`) y se leen los cubos. Como los cubos ya vienen agregados por la bodega (enfoque ROLAP), solo se transfieren unas pocas filas, no los 9 millones de viajes.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project='dw-taxi-nyc')

def leer(cubo):
    df = client.query(f'SELECT * FROM `dw-taxi-nyc.dw_taxi.{cubo}`').to_dataframe()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(float)
    return df

dem = leer('cubo_demanda')
ing = leer('cubo_ingresos')
op  = leer('cubo_operacion')
tie = leer('cubo_tiempo')

for n, d in [('cubo_demanda', dem), ('cubo_ingresos', ing), ('cubo_operacion', op), ('cubo_tiempo', tie)]:
    print(f'{n}: {len(d):,} filas')


## 2. Construcción de las visualizaciones
**Regla clave:** los cubos contienen filas de subtotal (`TOTAL`, `TODAS`, `TODOS`). Cada gráfico filtra exactamente el nivel de agregación que necesita, para no sumar dos veces el mismo viaje.

En los indicadores operativos por franja, los promedios se **ponderan por el número de viajes** de cada zona, en lugar de promediar promedios.

Decisiones visuales: barras para comparar categorías (la barra mayor se resalta), mapas de calor para cruces de dos dimensiones, serie de tiempo para la evolución diaria y tarjetas para las cifras principales.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- Estilo ----------
AZUL, CLARO, NARANJA = '#2a78d6', '#b7d1f2', '#eb6834'
FRANJAS = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
DIAS = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
CIUDAD = 'Toda la ciudad'


def num(n, dec=0):
    """Formato colombiano: 9.068.375 / 40,59"""
    s = f'{n:,.{dec}f}'
    return s.replace(',', 'X').replace('.', ',').replace('X', '.')


def resaltar(valores):
    """Barra más alta en azul fuerte, el resto en azul claro."""
    mx = max(valores)
    return [AZUL if v == mx else CLARO for v in valores]


def estilo(fig, titulo, alto=400):
    fig.update_layout(
        title=dict(text=titulo, x=0, xanchor='left', font=dict(size=16)),
        template='plotly_white', height=alto, showlegend=False,
        margin=dict(l=10, r=10, t=90, b=10),
        font=dict(family='Arial, sans-serif', size=12, color='#0b0b0b'),
        hoverlabel=dict(bgcolor='white', font_size=12),
        separators=',.', autosize=True,
    )
    fig.update_xaxes(showgrid=False)
    return fig


def selector(fig, etiquetas, trazas_por_opcion):
    """Menú desplegable que muestra solo las trazas de la opción elegida."""
    total = len(etiquetas) * trazas_por_opcion
    botones = []
    for i, e in enumerate(etiquetas):
        vis = [False] * total
        vis[i * trazas_por_opcion:(i + 1) * trazas_por_opcion] = [True] * trazas_por_opcion
        botones.append(dict(label=e, method='update', args=[{'visible': vis}]))
    fig.update_layout(updatemenus=[dict(
        buttons=botones, direction='down', showactive=True,
        x=1, xanchor='right', y=1.22, yanchor='top', bgcolor='white')])
    for k, tr in enumerate(fig.data):
        tr.visible = k < trazas_por_opcion
    return fig


def construir_dashboard(dem, ing, op, tie):
    figs, kpis, hallazgos = {}, {}, {}

    # ================= 1. DEMANDA (cubo_demanda) =================
    total = dem.query("borough=='TOTAL' and zona=='TODAS' and franja=='TODAS'").total_viajes.iloc[0]
    por_b = (dem.query("zona=='TODAS' and franja=='TODAS' and borough!='TOTAL'")
             .sort_values('total_viajes', ascending=False))
    boroughs = por_b.borough.tolist()
    opciones = [CIUDAD] + boroughs

    # Viajes por franja con selector de distrito
    f = go.Figure()
    for b in opciones:
        clave = 'TOTAL' if b == CIUDAD else b
        s = (dem[(dem.borough == clave) & (dem.zona == 'TODAS') & (dem.franja != 'TODAS')]
             .set_index('franja').reindex(FRANJAS).total_viajes.fillna(0))
        f.add_bar(x=s.index, y=s.values, marker_color=resaltar(s.values),
                  hovertemplate='%{x}: %{y:,.0f} viajes<extra></extra>')
    pico = (dem.query("borough=='TOTAL' and zona=='TODAS' and franja!='TODAS'")
            .sort_values('total_viajes').franja.iloc[-1])
    figs['dem_franja'] = selector(estilo(f, 'Viajes por franja horaria'), opciones, 1)

    # Viajes por distrito
    s = por_b.iloc[::-1]
    f = go.Figure(go.Bar(x=s.total_viajes, y=s.borough, orientation='h',
                         marker_color=resaltar(s.total_viajes.tolist()),
                         text=[num(v) for v in s.total_viajes], textposition='outside', cliponaxis=False,
                         hovertemplate='%{y}: %{x:,.0f} viajes<extra></extra>'))
    f.update_xaxes(range=[0, s.total_viajes.max() * 1.2])
    figs['dem_borough'] = estilo(f, 'Viajes por distrito de origen')

    # Mapa de calor: top 10 zonas × franja
    top = dem.query("zona!='TODAS' and franja=='TODAS'").nlargest(10, 'total_viajes')
    heat = (dem.query("zona!='TODAS' and franja!='TODAS'")
            .pivot_table(index='zona', columns='franja', values='total_viajes', aggfunc='sum')
            .reindex(index=top.zona, columns=FRANJAS).fillna(0))
    f = go.Figure(go.Heatmap(z=heat.values, x=FRANJAS, y=heat.index, colorscale='Blues',
                             text=[[num(v) for v in fila] for fila in heat.values],
                             texttemplate='%{text}', showscale=False,
                             hovertemplate='%{y} · %{x}: %{z:,.0f} viajes<extra></extra>'))
    f.update_yaxes(autorange='reversed')
    figs['dem_heat'] = estilo(f, 'Top 10 zonas de recogida por franja horaria', alto=480)

    pb = por_b.set_index('borough').total_viajes
    kpis['dem'] = [('Total de viajes', num(total)),
                   (f'Viajes desde {boroughs[0]}', f'{num(pb.iloc[0] / total * 100, 1)} %'),
                   ('Franja con más viajes', pico),
                   ('Zona con más viajes', str(top.zona.iloc[0]))]
    hallazgos['dem'] = (f'{boroughs[0]} y {boroughs[1]} concentran el '
                        f'{num((pb.iloc[0] + pb.iloc[1]) / total * 100, 1)} % de los viajes. '
                        f'La franja de mayor demanda es la {pico.lower()} y la zona con más recogidas es {top.zona.iloc[0]}. '
                        '<b>Acción sugerida:</b> priorizar la flota en aeropuertos y Midtown durante la tarde y la noche.')

    # ================= 2. INGRESOS (cubo_ingresos) =================
    tot = ing.query("borough=='TOTAL' and dia=='TODOS' and franja=='TODAS'").iloc[0]
    ticket = tot.ingresos_totales / tot.total_viajes

    f = go.Figure()
    for b in opciones:
        clave = 'TOTAL' if b == CIUDAD else b
        q = (ing[(ing.borough == clave) & (ing.franja == 'TODAS') & (ing.dia != 'TODOS')]
             .set_index('dia').reindex(DIAS))
        q = q.fillna(0)
        tk = (q.ingresos_totales / q.total_viajes.replace(0, pd.NA)).fillna(0)
        f.add_bar(x=q.index, y=q.ingresos_totales, marker_color=resaltar(q.ingresos_totales.tolist()),
                  customdata=list(zip(q.total_viajes, tk)),
                  hovertemplate='%{x}<br>Ingresos: USD %{y:,.0f}<br>Viajes: %{customdata[0]:,.0f}'
                                '<br>Ticket promedio: USD %{customdata[1]:.2f}<extra></extra>')
    f.update_yaxes(tickprefix='USD ')
    dias_ing = ing.query("borough=='TOTAL' and franja=='TODAS' and dia!='TODOS'").set_index('dia')
    mejor_dia, peor_dia = dias_ing.ingresos_totales.idxmax(), dias_ing.ingresos_totales.idxmin()
    figs['ing_dia'] = selector(estilo(f, 'Ingresos por día de la semana'), opciones, 1)

    heat = (ing.query("borough=='TOTAL' and dia!='TODOS' and franja!='TODAS'")
            .pivot_table(index='dia', columns='franja', values='ingresos_totales', aggfunc='sum')
            .reindex(index=DIAS, columns=FRANJAS))
    f = go.Figure(go.Heatmap(z=heat.values, x=FRANJAS, y=DIAS, colorscale='Blues', showscale=False,
                             text=[[f'{v / 1e6:.1f} M' for v in fila] for fila in heat.values],
                             texttemplate='%{text}',
                             hovertemplate='%{y} · %{x}: USD %{z:,.0f}<extra></extra>'))
    f.update_yaxes(autorange='reversed')
    figs['ing_heat'] = estilo(f, 'Ingresos por día y franja horaria (USD)', alto=420)

    s = (ing.query("borough!='TOTAL' and dia=='TODOS' and franja=='TODAS'")
         .sort_values('ingresos_totales'))
    f = go.Figure(go.Bar(x=s.ingresos_totales, y=s.borough, orientation='h',
                         marker_color=resaltar(s.ingresos_totales.tolist()),
                         text=[f'USD {v / 1e6:.1f} M' for v in s.ingresos_totales],
                         textposition='outside', cliponaxis=False,
                         hovertemplate='%{y}: USD %{x:,.0f}<extra></extra>'))
    f.update_xaxes(range=[0, s.ingresos_totales.max() * 1.25])
    figs['ing_borough'] = estilo(f, 'Ingresos por distrito de origen')

    kpis['ing'] = [('Ingresos totales', f'USD {num(tot.ingresos_totales / 1e6, 1)} M'),
                   ('Ticket promedio', f'USD {num(ticket, 2)}'),
                   ('Día de mayor ingreso', mejor_dia),
                   ('Día de menor ingreso', peor_dia)]
    hallazgos['ing'] = (f'El {mejor_dia.lower()} es el día de mayores ingresos y el {peor_dia.lower()} el de menores. '
                        f'El ticket promedio es de USD {num(ticket, 2)}. '
                        '<b>Acción sugerida:</b> concentrar la oferta entre semana y revisar incentivos para el fin de semana.')

    # ================= 3. OPERACIÓN (cubo_operacion) =================
    # Por distrito: cada distrito tiene su propia fila agregada
    ob = op.query("zona=='TODAS' and franja=='TODAS' and borough!='TOTAL'").set_index('borough')
    ob = ob.loc[[b for b in boroughs if b in ob.index]]
    metricas = [('distancia_promedio_millas', 'Distancia (millas)'),
                ('duracion_promedio_min', 'Duración (min)'),
                ('costo_promedio', 'Costo (USD)'),
                ('velocidad_promedio_mph', 'Velocidad (mph)')]
    f = make_subplots(rows=1, cols=4, shared_yaxes=True, subplot_titles=[m[1] for m in metricas],
                      horizontal_spacing=0.04)
    for c, (col, nom) in enumerate(metricas, start=1):
        f.add_bar(x=ob[col], y=ob.index, orientation='h', marker_color=AZUL, row=1, col=c,
                  hovertemplate='%{y}: %{x:.2f}<extra>' + nom + '</extra>')
    f.update_yaxes(autorange='reversed')
    figs['op_borough'] = estilo(f, 'Indicadores operativos por distrito de origen', alto=380)

    # Por franja: promedio ponderado por número de viajes (no promedio de promedios)
    det = op.query("zona!='TODAS' and franja!='TODAS'").copy()

    def ponderar(df):
        """Promedio ponderado por número de viajes de cada zona."""
        df = df.assign(dn=df.distancia_promedio_millas * df.total_viajes,
                       un=df.duracion_promedio_min * df.total_viajes,
                       cn=df.costo_promedio * df.total_viajes)
        s = df.groupby('franja')[['total_viajes', 'dn', 'un', 'cn']].sum()
        dur = s.un / s.total_viajes
        return pd.DataFrame({'Duración (min)': dur,
                             'Velocidad (mph)': (s.dn / s.total_viajes) / (dur / 60),
                             'Costo (USD)': s.cn / s.total_viajes}).reindex(FRANJAS)

    nombres = ['Duración (min)', 'Velocidad (mph)', 'Costo (USD)']
    f = make_subplots(rows=1, cols=3, subplot_titles=nombres, horizontal_spacing=0.07)
    for b in opciones:
        sub = det if b == CIUDAD else det[det.borough == b]
        w = ponderar(sub)
        for c, nom in enumerate(nombres, start=1):
            f.add_bar(x=w.index, y=w[nom], marker_color=AZUL, row=1, col=c,
                      hovertemplate='%{x}: %{y:.2f}<extra>' + nom + '</extra>')
    w_ciudad = ponderar(det)
    figs['op_franja'] = selector(estilo(f, 'Duración, velocidad y costo por franja horaria', alto=380),
                                 opciones, 3)

    gen = op.query("borough=='TOTAL'").iloc[0]
    lento = ob.velocidad_promedio_mph.idxmin()
    franja_lenta = w_ciudad['Velocidad (mph)'].idxmin()
    kpis['op'] = [('Distancia promedio', f'{num(gen.distancia_promedio_millas, 2)} mi'),
                  ('Duración promedio', f'{num(gen.duracion_promedio_min, 1)} min'),
                  ('Costo promedio', f'USD {num(gen.costo_promedio, 2)}'),
                  ('Velocidad promedio', f'{num(gen.velocidad_promedio_mph, 1)} mph')]
    hallazgos['op'] = (f'{lento} tiene la menor velocidad promedio ({num(ob.loc[lento, "velocidad_promedio_mph"], 1)} mph), '
                       f'señal de mayor congestión, y la {franja_lenta.lower()} es la franja más lenta de la ciudad. '
                       f'{ob.distancia_promedio_millas.idxmax()} registra los trayectos más largos. '
                       '<b>Acción sugerida:</b> considerar la congestión en la planificación de turnos y tiempos estimados.')

    # ================= 4. EVOLUCIÓN TEMPORAL (cubo_tiempo) =================
    dia = tie.query("nivel=='Día'").copy()
    dia['fecha'] = pd.to_datetime(dia.fecha)
    dia = dia.sort_values('fecha')
    f = go.Figure(go.Scatter(x=dia.fecha, y=dia.total_viajes, mode='lines', line=dict(color=AZUL, width=2),
                             hovertemplate='%{x|%d %b %Y}: %{y:,.0f} viajes<extra></extra>'))
    f.update_layout(hovermode='x unified')
    f.update_xaxes(tickformat='%b', rangeslider=dict(visible=True))
    figs['tie_dia'] = estilo(f, 'Viajes por día (2018) — arrastra la barra inferior para hacer zoom', alto=420)

    mes = tie.query("nivel=='Total mes'").sort_values('mes')
    mx, mn = mes.total_viajes.max(), mes.total_viajes.min()
    col = [AZUL if v == mx else NARANJA if v == mn else CLARO for v in mes.total_viajes]
    f = go.Figure(go.Bar(x=mes.nombre_mes, y=mes.total_viajes, marker_color=col,
                         hovertemplate='%{x}: %{y:,.0f} viajes<extra></extra>'))
    mes_max = mes.loc[mes.total_viajes.idxmax(), 'nombre_mes']
    mes_min = mes.loc[mes.total_viajes.idxmin(), 'nombre_mes']
    figs['tie_mes'] = estilo(f, f'Viajes por mes — pico en {mes_max} (azul), valle en {mes_min} (naranja)')

    tri = tie.query("nivel=='Total trimestre'").sort_values('trimestre')
    f = go.Figure(go.Bar(x=['T' + str(int(t)) for t in tri.trimestre], y=tri.total_viajes,
                         marker_color=resaltar(tri.total_viajes.tolist()),
                         hovertemplate='%{x}: %{y:,.0f} viajes<extra></extra>'))
    figs['tie_tri'] = estilo(f, 'Viajes por trimestre')

    kpis['tie'] = [('Promedio diario', f'{num(dia.total_viajes.mean())} viajes'),
                   ('Mes de mayor demanda', mes_max),
                   ('Mes de menor demanda', mes_min),
                   ('Diferencia pico–valle', f'{num((mx - mn) / mn * 100, 0)} %')]
    hallazgos['tie'] = (f'{mes_max} es el mes de mayor demanda y {mes_min} el de menor, con una diferencia del '
                        f'{num((mx - mn) / mn * 100, 0)} %. '
                        '<b>Acción sugerida:</b> ajustar el tamaño de la flota según la temporada.')
    return figs, kpis, hallazgos


def armar_html(figs, kpis, hallazgos):
    def g(k):
        return figs[k].to_html(full_html=False, include_plotlyjs=False,
                               config={'displaylogo': False, 'responsive': True})

    def tarjetas(lista):
        return '<div class="kpis">' + ''.join(
            f'<div class="kpi"><span>{t}</span><strong>{v}</strong></div>' for t, v in lista) + '</div>'

    secciones = [
        ('demanda', '1. Demanda de viajes', '¿En qué zonas y franjas horarias se concentran los viajes?',
         'dem', ['dem_franja', 'dem_borough'], ['dem_heat']),
        ('ingresos', '2. Análisis de ingresos', '¿Cómo varían los ingresos según la zona, el día y el horario?',
         'ing', ['ing_dia', 'ing_borough'], ['ing_heat']),
        ('operacion', '3. Operación de los viajes',
         '¿Cómo varían la distancia, la duración y el costo según la zona y la franja horaria?',
         'op', [], ['op_borough', 'op_franja']),
        ('tiempo', '4. Evolución temporal', '¿Cómo cambia la demanda a lo largo del tiempo?',
         'tie', ['tie_mes', 'tie_tri'], ['tie_dia']),
    ]
    cuerpo = ''
    for sid, titulo, pregunta, k, pares, anchos in secciones:
        cuerpo += f'<section id="{sid}"><h2>{titulo}</h2><p class="pregunta">{pregunta}</p>'
        cuerpo += tarjetas(kpis[k])
        cuerpo += f'<p class="hallazgo">{hallazgos[k]}</p>'
        if pares:
            cuerpo += '<div class="fila">' + ''.join(f'<div class="card">{g(x)}</div>' for x in pares) + '</div>'
        for x in anchos:
            cuerpo += f'<div class="card">{g(x)}</div>'
        cuerpo += '</section>'

    nav = ''.join(f'<a href="#{s[0]}">{s[1][3:]}</a>' for s in secciones)
    return f'''<!doctype html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Dashboard NYC Taxi 2018</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  body {{ margin:0; background:#f4f5f7; font-family:Arial, sans-serif; color:#0b0b0b; }}
  header {{ background:#13294b; color:white; padding:20px 24px; }}
  header h1 {{ margin:0; font-size:24px; }}
  header p {{ margin:6px 0 0; color:#c9d6ea; font-size:14px; }}
  nav {{ position:sticky; top:0; z-index:10; background:white; border-bottom:1px solid #ddd;
         display:flex; gap:4px; padding:8px 16px; overflow-x:auto; }}
  nav a {{ text-decoration:none; color:#13294b; padding:8px 14px; border-radius:6px; white-space:nowrap; font-size:14px; }}
  nav a:hover {{ background:#e8eef8; }}
  main {{ max-width:1300px; margin:auto; padding:16px; }}
  section {{ margin-bottom:40px; scroll-margin-top:60px; }}
  h2 {{ margin:16px 0 4px; font-size:21px; }}
  .pregunta {{ margin:0 0 14px; color:#52514e; font-style:italic; }}
  .kpis {{ display:grid; grid-template-columns:repeat(auto-fit, minmax(200px, 1fr)); gap:12px; margin-bottom:12px; }}
  .kpi {{ background:white; border-radius:10px; padding:14px 16px; border-left:4px solid #2a78d6; }}
  .kpi span {{ display:block; color:#52514e; font-size:13px; }}
  .kpi strong {{ display:block; font-size:22px; margin-top:4px; }}
  .hallazgo {{ background:#e8eef8; border-radius:10px; padding:12px 16px; margin:0 0 12px; line-height:1.5; }}
  .fila {{ display:grid; grid-template-columns:repeat(auto-fit, minmax(420px, 1fr)); gap:12px; }}
  .card {{ background:white; border-radius:10px; padding:8px; margin-bottom:12px; overflow:hidden; }}
  .card .js-plotly-plot, .card .plotly-graph-div {{ width:100% !important; }}
  footer {{ text-align:center; color:#52514e; font-size:12px; padding:20px; }}
</style></head>
<body>
<header><h1>Dashboard gerencial · Taxis de Nueva York 2018</h1>
<p>Fuente: bodega de datos en Google BigQuery (proyecto dw-taxi-nyc), cubos OLAP ROLAP · Use los menús desplegables de cada gráfico para filtrar por distrito.</p></header>
<nav>{nav}</nav>
<main>{cuerpo}</main>
<script>window.addEventListener('load', () => document.querySelectorAll('.js-plotly-plot').forEach(g => Plotly.Plots.resize(g)));</script>
<footer>Proyecto de aula Data Warehouse y Big Data · Politécnico Grancolombiano · 2026</footer>
</body></html>'''


## 3. Generación del dashboard
Se arma el tablero, se verifica que los totales cuadren con la bodega y se descarga como archivo HTML.

In [ ]:
figs, kpis, hallazgos = construir_dashboard(dem, ing, op, tie)

# Verificación: la suma por franja debe ser igual al total (sin duplicados)
total = dem.query("borough=='TOTAL' and zona=='TODAS' and franja=='TODAS'").total_viajes.iloc[0]
suma_franjas = dem.query("borough=='TOTAL' and zona=='TODAS' and franja!='TODAS'").total_viajes.sum()
print(f'Total de viajes: {total:,.0f} | Suma por franjas: {suma_franjas:,.0f} | Cuadra: {total == suma_franjas}')

html = armar_html(figs, kpis, hallazgos)
with open('dashboard_nyc_taxi_2018.html', 'w', encoding='utf-8') as f:
    f.write(html)

from google.colab import files
files.download('dashboard_nyc_taxi_2018.html')


## 4. Vista previa en Colab
Muestra cada gráfico dentro del notebook. Para las capturas del documento es mejor abrir el archivo HTML descargado en el navegador.

In [ ]:
for k, fig in figs.items():
    fig.show()
